In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
import numpy as np
import math
import seaborn as sns
import pyarrow as pa
import pyarrow.parquet as pq
import itertools
import evi_functions as evi_func
import matplotlib.dates as mdates
import warnings
from pathlib import Path
from datetime import datetime

import descri_function as des_fun

import functions_mem as fm
import functions  # auxiliary warning-cleaning utilities
import re

warnings.filterwarnings("ignore")


# Read Data

In [15]:
# Ler dado sintetico com os surtos definidos pelo MEM
df = pd.read_parquet('/opt/storage/shared/aesop/aesop_shared/ensamble_modelling/sintetic_with_MEM_surge_col_29_04_2026.parquet')

# Ler colunas com as anomalias identificadas em cada replica de cada modelo

ears = pd.read_parquet('/opt/storage/shared/aesop/aesop_shared/ensamble_modelling/EARS_municipios_bl8_alp005_sintetica.parquet')
maing = pd.read_parquet('/opt/storage/shared/aesop/aesop_shared/ensamble_modelling/outbreak_sintetic_mmaing_new.parquet')
#pd.read_parquet('/opt/storage/shared/aesop/aesop_shared/ensamble_modelling/outbreak_sintetic_mmaing.parquet')
evi = pd.read_parquet('/opt/storage/shared/aesop/aesop_shared/ensamble_modelling/sintetic_evi_07_05_2026.parquet')

dta = pd.read_parquet('/opt/storage/shared/aesop/aesop_shared/ensamble_modelling/cities_valid_for_MEM_26_03_2026.parquet')


# Formatar dado

In [16]:
ears["year_week"] = ears["ano"].astype(int).astype(str) + "-" + ears["epiweek"].astype(int).astype(str).str.zfill(2)

lst = list(set(ears.co_ibge.unique()) - set(dta.co_ibge.unique()))
ears = ears[~ears.co_ibge.isin(lst)]

ears =  ears[(ears.year_week >= '2022-42') &(ears.year_week <= '2025-32')]

series_list_ears = sorted(
    [col for col in ears.columns if col.startswith("C2_alarms_")],
    key=lambda x: int(x.split("_")[-1])
)

series_list_ears = ['co_ibge','year_week'] + series_list_ears
ears = ears[series_list_ears]

ears = ears.replace({'Não': 0, 'Sim': 1})

In [17]:
lst = list(set(maing.co_ibge.unique()) - set(dta.co_ibge.unique()))

maing= maing[~maing.co_ibge.isin(lst)]

maing =  maing[(maing.year_week >= '2022-42') &(maing.year_week <= '2025-32')]

series_list_maing1 = sorted(
    [col for col in maing.columns if col.startswith("EWS_ISF_replicate_")],
    key=lambda x: int(x.split("_")[-1])
)

series_list_maing2 = sorted(
    [col for col in maing.columns if col.startswith("EWS_LOF_replicate_")],
    key=lambda x: int(x.split("_")[-1])
)

series_list_maing3 = sorted(
    [col for col in maing.columns if col.startswith("EWS_OCSVM_replicate_")],
    key=lambda x: int(x.split("_")[-1])
)

series_list_maing4 = sorted(
    [col for col in maing.columns if col.startswith("EWS_COPOD_replicate_")],
    key=lambda x: int(x.split("_")[-1])
)

series_list_maing5 = sorted(
    [col for col in maing.columns if col.startswith("EWS_Rt_replicate_")],
    key=lambda x: int(x.split("_")[-1])
)

series_list_maing = ['co_ibge','year_week'] + series_list_maing1 + series_list_maing2 + series_list_maing3 + series_list_maing4 + series_list_maing5

maing = maing[series_list_maing]


In [18]:
lst = list(set(evi.co_ibge.unique()) - set(dta.co_ibge.unique()))

evi = evi[~evi.co_ibge.isin(lst)]

evi =  evi[(evi.year_week >= '2022-42') & (evi.year_week <= '2025-32')]

series_list_evi1 = [
    col for col in evi.columns
    if re.match(
        r"^mem_surge_01_replicate_\d+(_correct_with_consec)?$",
        col
    )
]

series_list_evi2 = sorted(
    [col for col in evi.columns if col.startswith("sinal_evi_replicate_")],
    key=lambda x: int(x.split("_")[-1])
)


series_list_evi = ['co_ibge','year_week',  'atend_ivas','mem_surge_01_correct_with_consec', 'warning_final_mem_surge_01'] + series_list_evi1 + series_list_evi2 

evi = evi[series_list_evi]


In [22]:
df_merged = evi.merge(ears, on=['co_ibge', 'year_week'], how='inner')

df_merged = df_merged.merge(maing, on=['co_ibge', 'year_week'], how='inner')

In [23]:
df_merged

,co_ibge,year_week,atend_ivas,mem_surge_01_correct_with_consec,warning_final_mem_surge_01,mem_surge_01_replicate_0,mem_surge_01_replicate_0_correct_with_consec,mem_surge_01_replicate_1,mem_surge_01_replicate_1_correct_with_consec,mem_surge_01_replicate_2,...,EWS_Rt_replicate_22,EWS_Rt_replicate_23,EWS_Rt_replicate_24,EWS_Rt_replicate_25,EWS_Rt_replicate_26,EWS_Rt_replicate_27,EWS_Rt_replicate_28,EWS_Rt_replicate_29,EWS_Rt_replicate_30,EWS_Rt_replicate_31
0,110001,2022-42,31,0,0,0,0,0,0,0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1,110001,2022-43,16,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,110001,2022-44,20,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,110001,2022-45,32,0,0,0,0,0,0,0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
4,110001,2022-46,47,0,0,0,0,0,0,0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
788650,530010,2025-28,9099,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
788651,530010,2025-29,8762,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
788652,530010,2025-30,7914,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
788653,530010,2025-31,7146,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [24]:
from pathlib import Path
from datetime import datetime

out_dir = Path("/opt/storage/shared/aesop/aesop_shared/ensamble_modelling")

fname = f"output_eadms_sintetic_{datetime.now():%d_%m_%Y}.parquet"

df_merged.to_parquet(out_dir / fname)